In [90]:
import pandas as pd
import numpy as np

In [91]:
df = pd.read_csv('/content/car_sales_messy.csv')


In [92]:
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())
print(df['car_model'].nunique())
print(df['dealer_city'].nunique())
df.head()

(49, 13)
['sale_id', 'date', 'car_make', 'car_model', 'model_year', 'price_usd', 'mileage_km', 'fuel_type', 'transmission', 'dealer_city', 'salesperson', 'customer_gender', 'payment_method']
sale_id             1
date                2
car_make            3
car_model           0
model_year          3
price_usd           0
mileage_km          9
fuel_type           1
transmission        2
dealer_city         1
salesperson         3
customer_gender    14
payment_method      2
dtype: int64
23
22


,sale_id,date,car_make,car_model,model_year,price_usd,mileage_km,fuel_type,transmission,dealer_city,salesperson,customer_gender,payment_method
0,S-1000,2024-06-24,NaN,Sonata,2021,$15737,56946 km,Diesel,Manual,Riyadh,Laila,NaN,Card
1,S-1001,15-Jul-2024,H0nd@,Civic,NaN,"$21,299","94,583",Diesel,Automatic,KHOBAR,Laila,Male,card
2,S-1002,01-26-2024,HONDA,Jazz,1899,"$14,348","59,189",Pertol,MANUAL,Dammam,Laila,Male,CASH
3,S-1003,03-08-2024,Honda,Jazz,2021,"$19,014","87,686",diesel,Automatic,madinah,Aisha,NaN,FINANCE
4,S-1004,2024/07/30,Kia,Sportage,2022,14716,"80,537",NaN,Automatic,NaN,Sara,NaN,Card


In [93]:
import numpy as np

df = df.replace(r'^\s*(N/A|na|-|)\s*$', np.nan, regex=True)

df['price_usd'] = pd.to_numeric(
    df['price_usd'].astype(str).str.replace(r'[$,]', '', regex=True),
    errors='coerce'
)

df['mileage_km'] = pd.to_numeric(
    df['mileage_km'].astype(str).str.replace(r'[,]|km', '', regex=True),
    errors='coerce'
)

df['model_year'] = pd.to_numeric(df['model_year'], errors='coerce')

df.loc[~df['model_year'].between(1900, 2025), 'model_year'] = np.nan

df = df.dropna(subset=['sale_id', 'date'])

for col in ['price_usd', 'mileage_km', 'model_year']:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Unknown')

In [94]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

for col in ['car_make', 'car_model', 'fuel_type', 'dealer_city', 'salesperson', 'customer_gender', 'payment_method']:
    df[col] = df[col].str.title()

df['transmission'] = df['transmission'].str.title()
df['transmission'] = df['transmission'].replace({
    'Auto': 'Automatic'
})

df['car_make'] = df['car_make'].replace({
    'H0nd@': 'Honda'
})

In [95]:
print(df.duplicated().sum())

df = df.drop_duplicates(subset=['sale_id'])

print(df.shape)

0
(44, 13)


In [96]:
df['date'] = pd.to_datetime(df['date'], errors='coerce', format='mixed')
df['model_year'] = pd.to_numeric(df['model_year'], errors='coerce')

df = df.dropna(subset=['sale_id', 'date'])

df['car_age'] = 2025 - df['model_year']

df['price_per_km'] = df['price_usd'] / df['mileage_km'].replace(0, np.nan)

print("Missing model_year:", df['model_year'].isna().sum())
print("Average car age:", df['car_age'].mean())
print(df[['date', 'model_year', 'car_age', 'price_per_km']].head())

Missing model_year: 0
Average car age: 5.976744186046512
        date  model_year  car_age  price_per_km
0 2024-06-24      2021.0      4.0      0.276350
1 2024-07-15      2019.0      6.0      0.225188
2 2024-01-26      2019.0      6.0      0.242410
3 2024-03-08      2021.0      4.0      0.216842
4 2024-07-30      2022.0      3.0      0.182723


In [97]:
print(df.describe())

print(df[['price_usd', 'mileage_km']].agg([
    'mean', 'median', 'std'
]))

print(df['car_make'].mode())

                                date   model_year     price_usd  \
count                             43    43.000000     43.000000   
mean   2024-07-26 01:06:58.604651264  2019.023256  13575.976744   
min              2024-01-01 00:00:00  2013.000000 -18312.000000   
25%              2024-04-17 12:00:00  2018.000000  12445.000000   
50%              2024-07-14 00:00:00  2019.000000  14716.000000   
75%              2024-11-09 12:00:00  2021.000000  16511.000000   
max              2025-02-11 00:00:00  2023.000000  22341.000000   
std                              NaN     2.511766   7782.277805   

          mileage_km    car_age  price_per_km  
count      43.000000  43.000000     43.000000  
mean    89852.860465   5.976744      0.217899  
min      8483.000000   2.000000     -0.206621  
25%     59990.000000   4.000000      0.118186  
50%     88626.000000   6.000000      0.175084  
75%    106603.500000   7.000000      0.259380  
max    195560.000000  12.000000      1.347636  
std     4100

In [98]:
print(df.groupby('car_make')['price_usd'].mean().sort_values(ascending=False))

print(df['dealer_city'].value_counts())

print(df.groupby('transmission')['price_usd'].mean())

car_make
H0Nd@      21299.000000
Unknown    18296.000000
Niss@N     17817.000000
Ford       14621.750000
Kia        13662.166667
Nissan     13141.200000
Hyundai    13043.333333
Honda      12617.416667
Toyota     11780.285714
Hyund@I    10596.000000
Name: price_usd, dtype: float64
dealer_city
Jeddah     10
Madinah    10
Riyadh      9
Dammam      8
Khobar      5
Unknown     1
Name: count, dtype: int64
transmission
Automatic    10859.800000
Manual       16235.111111
Unknown      14867.800000
Name: price_usd, dtype: float64


In [99]:
print(df.groupby('fuel_type')['price_usd'].mean())

print(df.groupby('salesperson')['price_usd'].sum().sort_values(ascending=False))

df['high_value_sale'] = df['price_usd'] > df['price_usd'].mean()

print(df.groupby('dealer_city')['high_value_sale'].mean().mul(100))

fuel_type
Diesel      13725.333333
Electric    13889.166667
Hybrid       9878.625000
Pertol      14932.500000
Petrol      16710.571429
Unknown     13262.500000
Name: price_usd, dtype: float64
salesperson
Laila      135450
Sara       130368
Aisha       90238
Khalid      88874
Unknown     53106
Omar        52127
Hassan      33604
Name: price_usd, dtype: int64
dealer_city
Dammam      25.000000
Jeddah      80.000000
Khobar      80.000000
Madinah     70.000000
Riyadh      66.666667
Unknown    100.000000
Name: high_value_sale, dtype: float64


In [100]:
df['car_make'] = df['car_make'].replace({
    'H0nd@': 'Honda',
    'Hyund@I': 'Hyundai',
    'Niss@N': 'Nissan'
})

df['fuel_type'] = df['fuel_type'].replace({
    'Pertol': 'Petrol'
})

df['transmission'] = df['transmission'].replace({
    'Auto': 'Automatic'
})

print(df['car_make'].value_counts())
print(df['fuel_type'].value_counts())
print(df.shape)

car_make
Honda      12
Toyota      7
Nissan      6
Kia         6
Ford        4
Hyundai     4
Unknown     3
H0Nd@       1
Name: count, dtype: int64
fuel_type
Diesel      12
Electric    12
Petrol       9
Hybrid       8
Unknown      2
Name: count, dtype: int64
(43, 16)


In [101]:
df['car_make'] = df['car_make'].replace({
    r'(?i)^h0nd@?$': 'Honda',
    r'(?i)^hyund@i$': 'Hyundai',
    r'(?i)^niss@n$': 'Nissan'
}, regex=True)

df['fuel_type'] = df['fuel_type'].replace({
    r'(?i)^pertol$': 'Petrol'
}, regex=True)

print(df['car_make'].value_counts())
print(df['fuel_type'].value_counts())
print(df.shape)

car_make
Honda      13
Toyota      7
Kia         6
Nissan      6
Ford        4
Hyundai     4
Unknown     3
Name: count, dtype: int64
fuel_type
Diesel      12
Electric    12
Petrol       9
Hybrid       8
Unknown      2
Name: count, dtype: int64
(43, 16)


In [102]:
df.to_csv('car_sales_cleaned.csv', index=False)

In [103]:

df['price_usd'] = pd.to_numeric(df['price_usd'], errors='coerce')
df['mileage_km'] = pd.to_numeric(df['mileage_km'], errors='coerce')
df['model_year'] = pd.to_numeric(df['model_year'], errors='coerce')

df.loc[df['price_usd'] <= 0, 'price_usd'] = np.nan
df.loc[df['mileage_km'] <= 0, 'mileage_km'] = np.nan
df.loc[~df['model_year'].between(1900, 2025), 'model_year'] = np.nan

for col in ['price_usd', 'mileage_km', 'model_year']:
    df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=['sale_id', 'date'])
df = df.drop_duplicates(subset=['sale_id'])

df['car_age'] = 2025 - df['model_year']
df['price_per_km'] = df['price_usd'] / df['mileage_km'].replace(0, np.nan)
df['high_value_sale'] = df['price_usd'] > df['price_usd'].mean()

print("Shape:", df.shape)
print("Duplicate sale_id:", df['sale_id'].duplicated().sum())
print("Missing values:")
print(df.isna().sum())
print("Non-positive prices:", (df['price_usd'] <= 0).sum())
print("Invalid mileage:", (df['mileage_km'] <= 0).sum())
print("Invalid model years:", (~df['model_year'].between(1900, 2025)).sum())
print("Car makes:")
print(df['car_make'].value_counts())
print("Fuel types:")
print(df['fuel_type'].value_counts())

df.to_csv('car_sales_cleaned.csv', index=False)


Shape: (43, 16)
Duplicate sale_id: 0
Missing values:
sale_id            0
date               0
car_make           0
car_model          0
model_year         0
price_usd          0
mileage_km         0
fuel_type          0
transmission       0
dealer_city        0
salesperson        0
customer_gender    0
payment_method     0
car_age            0
price_per_km       0
high_value_sale    0
dtype: int64
Non-positive prices: 0
Invalid mileage: 0
Invalid model years: 0
Car makes:
car_make
Honda      13
Toyota      7
Kia         6
Nissan      6
Ford        4
Hyundai     4
Unknown     3
Name: count, dtype: int64
Fuel types:
fuel_type
Diesel      12
Electric    12
Petrol       9
Hybrid       8
Unknown      2
Name: count, dtype: int64


In [104]:

print("Shape:", df.shape)
print("Duplicate sale_id:", df['sale_id'].duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:", df.isna().sum().sum())
print("Invalid prices:", (df['price_usd'] <= 0).sum())
print("Unknown car makes:", (df['car_make'] == 'Unknown').sum())
print("Unknown fuel types:", (df['fuel_type'] == 'Unknown').sum())


Shape: (43, 16)
Duplicate sale_id: 0
Duplicate rows: 0
Missing values: 0
Invalid prices: 0
Unknown car makes: 3
Unknown fuel types: 2


In [105]:
print("Unknown car makes:")
print(df[df['car_make'] == 'Unknown'])

print("Unknown fuel types:")
print(df[df['fuel_type'] == 'Unknown'])

Unknown car makes:
   sale_id       date car_make car_model  model_year  price_usd  mileage_km  \
0   S-1000 2024-06-24  Unknown    Sonata      2021.0    15737.0     56946.0   
17  S-1017 2024-06-16  Unknown    Escape      2022.0    22259.0     67239.0   
44  S-1044 2024-03-13  Unknown      Cr-V      2019.0    16892.0     30175.0   

   fuel_type transmission dealer_city salesperson customer_gender  \
0     Diesel       Manual      Riyadh       Laila         Unknown   
17    Petrol       Manual     Madinah      Khalid          Female   
44    Petrol    Automatic      Khobar      Khalid            Male   

   payment_method  car_age  price_per_km  high_value_sale  
0            Card      4.0      0.276350             True  
17        Unknown      3.0      0.331043             True  
44        Finance      6.0      0.559801             True  
Unknown fuel types:
   sale_id       date car_make car_model  model_year  price_usd  mileage_km  \
4   S-1004 2024-07-30      Kia  Sportage      20

In [106]:
original_df = pd.read_csv('/content/car_sales_messy.csv')

print(original_df.loc[[17, 44], :].to_string())
print(original_df.loc[[4, 28], :].to_string())

   sale_id        date car_make car_model model_year price_usd mileage_km fuel_type transmission dealer_city salesperson customer_gender payment_method
17  S-1017  06-16-2024      NaN    Escape       2022   $22,259      67239    Petrol       manual     MADINAH      Khalid          Female            NaN
44  S-1044  13/03/2024      NaN      CR-V       2019   $16,892   30175 km    Petrol         Auto      Khobar      Khalid            Male        Finance
   sale_id        date car_make car_model model_year price_usd mileage_km fuel_type transmission dealer_city salesperson customer_gender payment_method
4   S-1004  2024/07/30     Kia   Sportage       2022     14716     80,537       NaN    Automatic         NaN        Sara             NaN          Card 
28  S-1028  07-10-2024    Honda    Accord       2017     11809        NaN         -                  madinah       Laila            Male           Card


In [107]:
cols = [
    'sale_id', 'car_make', 'car_model',
    'fuel_type', 'transmission', 'dealer_city'
]

print("Unknown car makes:")
print(df.loc[df['car_make'] == 'Unknown', cols].to_string(index=False))

print("\nUnknown fuel types:")
print(df.loc[df['fuel_type'] == 'Unknown', cols].to_string(index=False))

Unknown car makes:
sale_id car_make car_model fuel_type transmission dealer_city
 S-1000  Unknown    Sonata    Diesel       Manual      Riyadh
 S-1017  Unknown    Escape    Petrol       Manual     Madinah
 S-1044  Unknown      Cr-V    Petrol    Automatic      Khobar

Unknown fuel types:
sale_id car_make car_model fuel_type transmission dealer_city
 S-1004      Kia  Sportage   Unknown    Automatic     Unknown
 S-1028    Honda    Accord   Unknown      Unknown     Madinah


In [108]:
df.loc[df['sale_id'] == 'S-1000', 'car_make'] = 'Hyundai'
df.loc[df['sale_id'] == 'S-1017', 'car_make'] = 'Ford'
df.loc[df['sale_id'] == 'S-1044', 'car_make'] = 'Honda'

print(df[df['car_make'] == 'Unknown'])

Empty DataFrame
Columns: [sale_id, date, car_make, car_model, model_year, price_usd, mileage_km, fuel_type, transmission, dealer_city, salesperson, customer_gender, payment_method, car_age, price_per_km, high_value_sale]
Index: []


In [109]:
print(
    df.loc[
        df['fuel_type'] == 'Unknown',
        ['sale_id', 'car_make', 'car_model', 'model_year', 'fuel_type']
    ].to_string(index=False)
)

sale_id car_make car_model  model_year fuel_type
 S-1004      Kia  Sportage      2022.0   Unknown
 S-1028    Honda    Accord      2017.0   Unknown


In [110]:
print(
    df.loc[
        df['sale_id'].isin(['S-1004', 'S-1028'])
    ].to_string(index=False)
)

sale_id       date car_make car_model  model_year  price_usd  mileage_km fuel_type transmission dealer_city salesperson customer_gender payment_method  car_age  price_per_km  high_value_sale
 S-1004 2024-07-30      Kia  Sportage      2022.0    14716.0     80537.0   Unknown    Automatic     Unknown        Sara         Unknown           Card      3.0      0.182723            False
 S-1028 2024-07-10    Honda    Accord      2017.0    11809.0     88626.0   Unknown      Unknown     Madinah       Laila            Male           Card      8.0      0.133245            False


In [111]:
original_df = pd.read_csv('/content/car_sales_messy.csv')

print(
    original_df.loc[
        original_df['sale_id'].isin(['S-1004', 'S-1028'])
    ].to_string(index=False)
)

sale_id       date car_make car_model model_year price_usd mileage_km fuel_type transmission dealer_city salesperson customer_gender payment_method
 S-1004 2024/07/30     Kia   Sportage       2022     14716     80,537       NaN    Automatic         NaN        Sara             NaN          Card 
 S-1028 07-10-2024    Honda    Accord       2017     11809        NaN         -                  madinah       Laila            Male           Card


In [112]:
print("Missing values in original data:")
print(original_df.isnull().sum())

print("\nDash values in original data:")
print((original_df.astype(str) == '-').sum())

Missing values in original data:
sale_id             1
date                2
car_make            3
car_model           0
model_year          3
price_usd           0
mileage_km          9
fuel_type           1
transmission        2
dealer_city         1
salesperson         3
customer_gender    14
payment_method      2
dtype: int64

Dash values in original data:
sale_id            1
date               0
car_make           0
car_model          0
model_year         3
price_usd          0
mileage_km         1
fuel_type          2
transmission       0
dealer_city        0
salesperson        1
customer_gender    3
payment_method     1
dtype: int64


In [113]:
print("Current cleaned data:")
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nUnknown car makes:")
print(df[df['car_make'] == 'Unknown'][['sale_id', 'car_make']])
print("\nUnknown fuel types:")
print(df[df['fuel_type'] == 'Unknown'][['sale_id', 'fuel_type']])

Current cleaned data:
Shape: (43, 16)

Missing values:
sale_id            0
date               0
car_make           0
car_model          0
model_year         0
price_usd          0
mileage_km         0
fuel_type          0
transmission       0
dealer_city        0
salesperson        0
customer_gender    0
payment_method     0
car_age            0
price_per_km       0
high_value_sale    0
dtype: int64

Unknown car makes:
Empty DataFrame
Columns: [sale_id, car_make]
Index: []

Unknown fuel types:
   sale_id fuel_type
4   S-1004   Unknown
28  S-1028   Unknown


In [114]:
print("Duplicate sale_id:",
      df['sale_id'].duplicated().sum())

print("Duplicate rows:",
      df.duplicated().sum())

print("Invalid prices:",
      (df['price_usd'] <= 0).sum())

print("Invalid mileage:",
      (df['mileage_km'] < 0).sum())

print("Invalid model years:",
      ((df['model_year'] < 1900) |
       (df['model_year'] > 2026)).sum())

print("Final shape:", df.shape)

Duplicate sale_id: 0
Duplicate rows: 0
Invalid prices: 0
Invalid mileage: 0
Invalid model years: 0
Final shape: (43, 16)


In [115]:
df.to_csv('/content/car_sales_cleaned.csv', index=False)

print("Saved successfully!")
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())

Saved successfully!
Shape: (43, 16)
Missing values: 0
